#### **UAE Fleet Inventory, Purchase Orders & Backorder Data Generation**

This notebook invokes the modular generator in `src/data_generator.py` to create realistic, cross-linked datasets for Fleet Spare-Parts Inventory Replenishment & Safety Stock Optimiser.

**Key Guardrails:**
* All operational dates and expected receipt dates are strictly capped within 2025.
* All financial figures (`unit_cost_aed`, `po_value_aed`) are denominated in AED.
* Stock positions explicitly separate On-Hand, Due-In, Reserved, and Unserviceable repairable cores.

In [1]:
import sys
from pathlib import Path

# Add project root to sys.path so `src` imports cleanly
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_generator import generate_inventory_datasets

datasets = generate_inventory_datasets(
    output_dir=str(PROJECT_ROOT / "data" / "raw"),
    seed=42,
    snapshot_date_str="2025-12-15",
)

inv_df = datasets["inventory_snapshot"]
po_df = datasets["open_purchase_orders"]
bo_df = datasets["backordered_work_orders"]

print(f"Snapshot rows: {len(inv_df)} | Open PO rows: {len(po_df)} | Backordered WOs: {len(bo_df)}")

Snapshot rows: 28 | Open PO rows: 5 | Backordered WOs: 15


#### **Schema & Data Quality Audit**

In [3]:
print("== Inventory Snapshot Preview ==")
display(inv_df.head(5))

print("\n== Missing Values & Data Types ==")
print(inv_df.isna().sum()[inv_df.isna().sum() > 0])
print("\nCriticality breakdown:\n", inv_df["criticality"].value_counts())
print("\nHub breakdown:\n", inv_df["hub_location"].value_counts())

== Inventory Snapshot Preview ==


,snapshot_date,part_number,part_name,category,vehicle_type,hub_location,supplier,on_hand_qty,due_in_qty,expected_receipt_date,po_status,reserved_qty,unserviceable_qty,avg_monthly_demand,demand_std_dev,lead_time_days,unit_cost_aed,criticality,last_issue_date,months_since_last_issue
0,2025-12-15,PRT-001,Heavy-Duty Engine Oil Filter,Filters & Fluids,Heavy Truck,Jebel Ali Hub,Al-Futtaim Auto Parts UAE,46,0,N/A,No Open PO,4,1,48.0,9.5,14,95.0,High,2025-12-15,0
1,2025-12-15,PRT-002,Primary Fuel Water Separator,Filters & Fluids,Heavy Truck,Jebel Ali Hub,Emirates Fleet Supply,6,17,2025-12-15,In Transit,6,1,34.0,8.2,18,165.0,High,2025-12-15,0
2,2025-12-15,PRT-003,Cabin Air Filter Element,Filters & Fluids,Light Commercial Van,Sajaa Hub,Gulf Auto Components,25,0,N/A,No Open PO,3,1,42.0,7.0,10,55.0,Low,2025-12-15,0
3,2025-12-15,PRT-004,Heavy-Duty Air Intake Filter,Filters & Fluids,Medium Truck,Mussafah Hub,Al-Futtaim Auto Parts UAE,18,0,N/A,No Open PO,2,0,26.0,6.4,15,210.0,Medium,2025-12-15,0
4,2025-12-15,PRT-005,Front Disc Brake Pad Set,Braking System,Heavy Truck,Jebel Ali Hub,Arabian Brake & Axle Co.,6,15,2025-12-26,In Transit,6,1,30.0,7.5,21,480.0,High,2025-12-15,0



== Missing Values & Data Types ==
Series([], dtype: int64)

Criticality breakdown:
 criticality
High      16
Medium     8
Low        4
Name: count, dtype: int64

Hub breakdown:
 hub_location
Jebel Ali Hub    11
Mussafah Hub      9
Sajaa Hub         8
Name: count, dtype: int64


#### **Date Cap Verification (<= 2025-12-31) & Pipeline Audit (open_purchase_orders.csv)**

In [4]:
# Verify all snapshot and last_issue_date values are in 2025
assert (inv_df["snapshot_date"].str[:4] == "2025").all(), "Snapshot date outside 2025!"
assert (inv_df["last_issue_date"].str[:4] == "2025").all(), "Last issue date outside 2025!"

# Verify all PO dates are within 2025
assert (po_df["order_date"].str[:4] == "2025").all(), "Order date outside 2025!"
assert (po_df["expected_receipt_date"].str[:4] == "2025").all(), "Receipt date outside 2025!"

print("2025 date cap checks passed across all tables.")
print("\n=== Open Purchase Orders by Status ===")
print(po_df.groupby("po_status", as_index=False).agg(
    po_count=("po_number", "count"),
    total_units=("due_in_qty", "sum"),
    total_value_aed=("po_value_aed", "sum"),
))

2025 date cap checks passed across all tables.

=== Open Purchase Orders by Status ===
    po_status  po_count  total_units  total_value_aed
0  In Transit         4           48          14180.0
1     Overdue         1            9           3060.0


#### **Delayed Work Orders Summary (backordered_work_orders.csv)**

In [6]:
print("== Backordered Vehicle Work Orders Preview ==")
display(bo_df.head(5))

wo_summary = (
    bo_df.groupby(["vehicle_type", "hub_location"], as_index=False)
    .agg(
        delayed_wos=("work_order_id", "count"),
        total_delay_days=("days_delayed", "sum"),
        total_downtime_hrs=("downtime_hours_accumulated", "sum"),
        total_short_qty=("qty_required", "sum"),
    )
    .sort_values("total_downtime_hrs", ascending=False)
)
print("\n== Downtime Impact by Vehicle Class & Hub ==")
display(wo_summary)

== Backordered Vehicle Work Orders Preview ==


,work_order_id,vehicle_id,vehicle_type,hub_location,part_number,part_name,criticality,wo_opened_date,snapshot_date,days_delayed,downtime_hours_accumulated,qty_required,delay_reason
0,WO-2025-8401,DXB-H-4088,Heavy Truck,Jebel Ali Hub,PRT-002,Primary Fuel Water Separator,High,2025-12-10,2025-12-15,5,42.5,3,Awaiting Spare Part Stock Replenishment
1,WO-2025-8402,DXB-H-4012,Heavy Truck,Jebel Ali Hub,PRT-005,Front Disc Brake Pad Set,High,2025-12-03,2025-12-15,12,102.0,1,Awaiting Spare Part Stock Replenishment
2,WO-2025-8403,AUH-H-7120,Heavy Truck,Jebel Ali Hub,PRT-005,Front Disc Brake Pad Set,High,2025-12-02,2025-12-15,13,110.5,2,Awaiting Spare Part Stock Replenishment
3,WO-2025-8404,AUH-H-7120,Heavy Truck,Jebel Ali Hub,PRT-005,Front Disc Brake Pad Set,High,2025-12-10,2025-12-15,5,42.5,1,Awaiting Spare Part Stock Replenishment
4,WO-2025-8405,DXB-B-9015,Coach / Bus,Jebel Ali Hub,PRT-007,ABS Wheel Speed Sensor,High,2025-12-01,2025-12-15,14,119.0,1,Awaiting Spare Part Stock Replenishment



== Downtime Impact by Vehicle Class & Hub ==


,vehicle_type,hub_location,delayed_wos,total_delay_days,total_downtime_hrs,total_short_qty
2,Heavy Truck,Jebel Ali Hub,10,78,663.0,20
1,Coach / Bus,Mussafah Hub,2,20,170.0,5
0,Coach / Bus,Jebel Ali Hub,1,14,119.0,1
4,Medium Truck,Mussafah Hub,1,8,68.0,1
3,Heavy Truck,Mussafah Hub,1,7,59.5,3
